# 04 Modern Segmentation Baseline: PyTorch U-Net for Warp & Weft Semantic Segmentation

## 1. Mathematical Formulation & Architecture
This notebook trains and evaluates a modern deep learning semantic segmentation baseline for fabric microscopy using PyTorch.

### Theoretical Problem Formulation
Given a fabric microscope image $I \in \mathbb{R}^{H \times W}$, our objective is to assign each pixel $(y, x)$ to one of three semantic classes:
- Class 0: Interstitial void / background ($c=0$)
- Class 1: Longitudinal warp yarn core ($c=1$)
- Class 2: Transverse weft yarn core ($c=2$)

### Architectural Specifications:
- Model: `WarpWeftUNet` (encoder-decoder with skip connections)
- Framework: PyTorch 2.x
- Loss Function: Multi-class Cross-Entropy Loss
$$\mathcal{L}_{CE} = -\frac{1}{HW} \sum_{h=1}^H \sum_{w=1}^W \sum_{c=0}^2 y_{h,w,c} \log \hat{p}_{h,w,c}$$
- Evaluation Metrics: Mean Intersection-over-Union (mIoU) and pixel accuracy.

In [1]:
# The mechanistic justification for this step is: initialize PyTorch, CUDA device contexts, and deterministic seeds to ensure reproducible neural training.
import os
import time
import json
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Compute Device:", device)
if torch.cuda.is_available():
    print("GPU Model:", torch.cuda.get_device_name(0))


Compute Device: cuda:0
GPU Model: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [2]:
# The mechanistic justification for this step is: synthesize micro-swatches with mathematically exact pixel-level warp/weft ground truth masks under realistic optical variations (yarn blur, fiber hairiness, and sensor noise).
class SyntheticFabricGenerator:
    @staticmethod
    def generate_sample(width=256, height=256, warp_count=16, weft_count=14, weave_type='plain', noise_sigma=0.08):
        # Physical grids
        x = np.linspace(0, width, width, endpoint=False)
        y = np.linspace(0, height, height, endpoint=False)
        xx, yy = np.meshgrid(x, y)

        warp_pitch = width / float(warp_count)
        weft_pitch = height / float(weft_count)

        # Warp phase and profile
        warp_phase = (xx % warp_pitch) / warp_pitch
        weft_phase = (yy % weft_pitch) / weft_pitch

        warp_profile = np.sin(np.pi * warp_phase) ** 2
        weft_profile = np.sin(np.pi * weft_phase) ** 2

        # Weave float logic
        warp_idx = (xx // warp_pitch).astype(int)
        weft_idx = (yy // weft_pitch).astype(int)

        if weave_type == 'plain':
            warp_on_top = ((warp_idx + weft_idx) % 2 == 0)
        elif weave_type == 'twill':
            warp_on_top = ((warp_idx + weft_idx) % 3 < 2)
        else: # satin 4/1
            warp_on_top = ((warp_idx + 2 * weft_idx) % 5 == 0)

        # Construct image intensity
        base_intensity = 0.5
        warp_signal = 0.4 * warp_profile
        weft_signal = 0.4 * weft_profile

        composite = np.where(warp_on_top,
                             base_intensity + warp_signal - 0.2 * (1.0 - weft_profile),
                             base_intensity + weft_signal - 0.2 * (1.0 - warp_profile))

        # Add Gaussian blur to simulate lens PSF
        blurred = cv2.GaussianBlur(composite.astype(np.float32), (5, 5), 1.2)

        # Add sensor noise
        noise = np.random.normal(0, noise_sigma, (height, width))
        image = np.clip(blurred + noise, 0.0, 1.0).astype(np.float32)

        # Construct exact ground truth mask
        # 0: interstice/void, 1: warp, 2: weft
        mask = np.zeros((height, width), dtype=np.int64)
        interstice = (warp_profile < 0.2) & (weft_profile < 0.2)

        mask[~interstice & warp_on_top] = 1
        mask[~interstice & ~warp_on_top] = 2
        mask[interstice] = 0

        return image, mask

class FabricPatchDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img, mask = self.samples[idx]
        tensor_img = torch.from_numpy(img).unsqueeze(0).float() # (1, H, W)
        tensor_mask = torch.from_numpy(mask).long()            # (H, W)
        return tensor_img, tensor_mask

print("Generating synthetic micro-swatch dataset...")
all_samples = []
weave_choices = ['plain', 'twill', 'satin']
for i in range(120):
    w_count = int(np.random.randint(12, 24))
    wf_count = int(np.random.randint(10, 22))
    w_type = weave_choices[i % len(weave_choices)]
    im, mk = SyntheticFabricGenerator.generate_sample(256, 256, w_count, wf_count, w_type)
    all_samples.append((im, mk))

train_samples = all_samples[:80]
val_samples = all_samples[80:100]
test_samples = all_samples[100:]

train_ds = FabricPatchDataset(train_samples)
val_ds = FabricPatchDataset(val_samples)
test_ds = FabricPatchDataset(test_samples)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False)

print(f"Generated {len(train_ds)} train, {len(val_ds)} val, and {len(test_ds)} test patches.")


Generating synthetic micro-swatch dataset...


Generated 80 train, 20 val, and 20 test patches.


In [3]:
# The mechanistic justification for this step is: construct a lightweight PyTorch U-Net with skip connections to preserve high-resolution spatial boundaries between adjacent yarn filaments.
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.1, inplace=True)
        )

    def forward(self, x):
        return self.net(x)

class WarpWeftUNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=3):
        super().__init__()
        self.inc = DoubleConv(in_channels, 32)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(32, 64))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))

        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(128, 64)

        self.up3 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(64, 32)

        self.outc = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)

        u1 = self.up1(x4)
        u1 = self.conv_up1(torch.cat([u1, x3], dim=1))

        u2 = self.up2(u1)
        u2 = self.conv_up2(torch.cat([u2, x2], dim=1))

        u3 = self.up3(u2)
        u3 = self.conv_up3(torch.cat([u3, x1], dim=1))

        logits = self.outc(u3)
        return logits

model = WarpWeftUNet(in_channels=1, num_classes=3).to(device)
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Initialized WarpWeftUNet on {device}. Trainable parameters: {param_count:,}")


Initialized WarpWeftUNet on cuda:0. Trainable parameters: 1,926,499


In [4]:
# The mechanistic justification for this step is: train the U-Net over 15 epochs, tracking CrossEntropy loss and computing mean Intersection-over-Union (mIoU) on validation patches.
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

def compute_miou(preds, targets, num_classes=3):
    ious = []
    preds = preds.view(-1)
    targets = targets.view(-1)
    for c in range(num_classes):
        pred_c = (preds == c)
        target_c = (targets == c)
        intersection = (pred_c & target_c).sum().item()
        union = (pred_c | target_c).sum().item()
        if union == 0:
            ious.append(float('nan'))
        else:
            ious.append(intersection / union)
    return np.nanmean(ious)

epochs = 15
history = []

print(f"{'Epoch':<6} | {'Train Loss':<11} | {'Val Loss':<10} | {'Val mIoU':<10} | {'Elapsed (s)':<10}")
print("-" * 55)

best_val_miou = 0.0
checkpoint_save_path = "experiments/learned_models/checkpoints/unet_yarn_segmentation.pt"
os.makedirs("experiments/learned_models/checkpoints", exist_ok=True)

for epoch in range(1, epochs + 1):
    t0 = time.time()
    model.train()
    train_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_ds)

    # Validation
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            loss = criterion(logits, masks)
            val_loss += loss.item() * imgs.size(0)
            preds = torch.argmax(logits, dim=1)
            all_preds.append(preds.cpu())
            all_targets.append(masks.cpu())
    val_loss /= len(val_ds)

    val_preds = torch.cat(all_preds, dim=0)
    val_targets = torch.cat(all_targets, dim=0)
    val_miou = compute_miou(val_preds, val_targets)
    elapsed = time.time() - t0

    history.append({
        "epoch": epoch,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "val_miou": float(val_miou),
        "time_s": float(elapsed)
    })

    print(f"{epoch:<6} | {train_loss:<11.4f} | {val_loss:<10.4f} | {val_miou:<10.4f} | {elapsed:<10.2f}")

    if val_miou > best_val_miou:
        best_val_miou = val_miou
        torch.save(model.state_dict(), checkpoint_save_path)

print(f"\nBest Validation mIoU: {best_val_miou:.4f}. Saved checkpoint to {checkpoint_save_path}")


Epoch  | Train Loss  | Val Loss   | Val mIoU   | Elapsed (s)
-------------------------------------------------------


1      | 0.6971      | 1.0583     | 0.2379     | 2.91      


2      | 0.3456      | 0.8270     | 0.2406     | 1.75      


3      | 0.2686      | 0.3841     | 0.5786     | 1.71      


4      | 0.2311      | 0.2707     | 0.6802     | 1.70      


5      | 0.1866      | 0.2036     | 0.7940     | 1.78      


... (6 outputs left out)

12     | 0.1102      | 0.1359     | 0.8765     | 1.72      


13     | 0.1118      | 0.1564     | 0.8546     | 1.80      


14     | 0.0946      | 0.0819     | 0.9068     | 1.74      


15     | 0.0890      | 0.0717     | 0.9258     | 1.76      

Best Validation mIoU: 0.9258. Saved checkpoint to experiments/learned_models/checkpoints/unet_yarn_segmentation.pt


In [5]:
# The mechanistic justification for this step is: evaluate the trained checkpoint on unseen test patches and calculate per-class IoU and pixel accuracy.
model.load_state_dict(torch.load(checkpoint_save_path, map_location=device))
model.eval()

test_preds = []
test_targets = []
with torch.no_grad():
    for imgs, masks in test_loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        preds = torch.argmax(logits, dim=1).cpu()
        test_preds.append(preds)
        test_targets.append(masks)

all_test_preds = torch.cat(test_preds, dim=0).view(-1)
all_test_targets = torch.cat(test_targets, dim=0).view(-1)

pixel_accuracy = (all_test_preds == all_test_targets).float().mean().item()

class_names = ["Background / Void", "Warp Yarn Core", "Weft Yarn Core"]
class_ious = []
for c in range(3):
    p_c = (all_test_preds == c)
    t_c = (all_test_targets == c)
    inter = (p_c & t_c).sum().item()
    union = (p_c | t_c).sum().item()
    iou = inter / union if union > 0 else 0.0
    class_ious.append(iou)

test_miou = float(np.mean(class_ious))

print("\n=== TEST SET SEGMENTATION PERFORMANCE ===")
print(f"Overall Pixel Accuracy: {pixel_accuracy * 100:.2f}%")
print(f"Overall Mean IoU (mIoU): {test_miou * 100:.2f}%\n")
for name, iou in zip(class_names, class_ious):
    print(f"Class IoU [{name}]: {iou * 100:.2f}%")

# Save results to json
test_results = {
    "model_name": "WarpWeftUNet",
    "checkpoint": checkpoint_save_path,
    "parameter_count": param_count,
    "best_val_miou": float(best_val_miou),
    "test_pixel_accuracy": float(pixel_accuracy),
    "test_miou": float(test_miou),
    "class_ious": {name: float(iou) for name, iou in zip(class_names, class_ious)},
    "history": history
}

with open("experiments/learned_models/results/unet_training_results.json", "w") as f:
    json.dump(test_results, f, indent=2)

print("\nResults successfully saved to experiments/learned_models/results/unet_training_results.json")



=== TEST SET SEGMENTATION PERFORMANCE ===
Overall Pixel Accuracy: 97.82%
Overall Mean IoU (mIoU): 91.81%

Class IoU [Background / Void]: 81.16%
Class IoU [Warp Yarn Core]: 96.80%
Class IoU [Weft Yarn Core]: 97.48%

Results successfully saved to experiments/learned_models/results/unet_training_results.json
